In [8]:
import pandas as pd

DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


Pandas loaded 144,724 rows and 15 columns with the default settings. 

In [9]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)  # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


The default load gave 144,724 rows. Loading with quoting=3 gave 145,431 rows. That is a difference of 707 rows. Pandas did not show any warning or error, so the missing rows were lost silently. 

Pandas did not complain, but the default settings dropped 707 reviews. Stray quote marks in the text made pandas read some reviews as part of the next row, so rows got merged together. Code that runs without errors can still give you wrong data. You have to check the row count against what you expect, and compare different loading settings, before you trust any results built on top of it. 

In [10]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

Several reviews contain HTML leftovers. I saw <br /> line break tags inside sentences, &#34; codes where quote marks should be, and \\" escape characters in one review. There are also typos and uneven spacing.

The reviews were written on a website that stores text as HTML. When the data was exported, the HTML formatting was saved along with the text instead of being converted to plain characters. 

In [25]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171700
2    0.053277
3    0.079964
4    0.140317
5    0.554742
Name: proportion, dtype: float64

Most reviews are positive. About 55% are 5 stars and about 70% are 4 or 5 stars. Only about 5% are 2 stars and 8% are 3 stars. This matters because a model that said “positive” for every review would still be right about 70% of the time without understanding anything. A high accuracy score on this data could just reflect the imbalance, not a good model. 

In [12]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

Many of the most reviewed items are not games at all. They are gift cards, points cards, and subscriptions. About 22% of all reviews are for these five products. Reviews of a gift card talk about delivery, redemption codes, or customer service, not gameplay. So the dataset is not purely game reviews, and a sentiment model tested on it is partly being tested on reviews about codes and billing. 

In [13]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

In [14]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


In [15]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

C:\Users\abarmecha1\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abarmecha1\.cache\huggingface\hub\models--distilbert--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

In [16]:
%%time
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

CPU times: total: 4min 25s
Wall time: 36.6 s


,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


In [17]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"

sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


In [18]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


In [19]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



In [20]:
for review in mistakes[mistakes["star_rating"] >= 4].head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

4 NEGATIVE 0.99
This is not technically a review, as such, but a reevaluation. Crytek just released the v1.9 patch which adds not only full DirectX 11 support, but also 2 separately installable packs - one for the new \\"Ultra\\" setting, and a specific \\"High-Resolution Pack\\" incorporating DirectX 11. You can f 

4 NEGATIVE 0.91
I was worried to get this for my husband for Christmas given some of the concerns regarding downloading I had read in the reviews, but for the value I couldn't NOT try it.  Didn't have a problem in the world with the download.  The only gripe I would have is that it took 6+ hours to actually load, b 

4 NEGATIVE 0.98
scary and def cool game glad I bought it now need to solve this game after fig it out lol 

4 NEGATIVE 0.92
I really like this expansion pack.  The only issues I have are with the time length of visiting each location.  If your sim hasn't traveled much they are limited to three days at any of the locations.  This doesn't really give you a lot o

In [21]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = star_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

C:\Users\abarmecha1\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abarmecha1\.cache\huggingface\hub\models--nlptown--bert-base-multilingual-uncased-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\abarmecha1\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_do

model.safetensors: reconstructing file:   0%|          |  0.00B /  669MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Exact: 0.574
Within one star: 0.912


In [22]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)
complaints["topic"] = [r["labels"][0] for r in results]
complaints["topic"].value_counts()

config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

C:\Users\abarmecha1\hf-explore\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\abarmecha1\.cache\huggingface\hub\models--facebook--bart-large-mnli. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

In [23]:
for _, row in complaints.head(3).iterrows():
    print(row["topic"], "|", row["review_body"][:300], "\n")

price | Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not). 

download or install | I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is. 

gameplay | If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\". 



In [24]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 

